In [1]:
from src.utils import AIDatasetLoader, filter_by_app_and_model, DecisionTreeInterpreter, LogisticRegressionInterpreter  
from src.cognitive_models.memory import Chunk, DeclarativeMemory

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [4]:
current_dir = os.getcwd()
data_dir = os.path.join(current_dir, 'datasets')

file_values = os.path.join(data_dir, 'values.csv')
file_metadata = os.path.join(data_dir, 'metadata.csv')
file_prediction = os.path.join(data_dir, 'none.csv')

values_df = pd.read_csv(file_values)
metadata_df = pd.read_csv(file_metadata)
prediction_df = pd.read_csv(file_prediction)

# ✅ Which model to use per dataset
dataset_model_map = {
    "mushrooms": "mlp",
    "wine_quality": "mlp",
    "forest_cover": "xgboost",
    "adult": "xgboost",
}

# # 🔧 Choose dataset here:
app_id = "wine_quality"  # 🔄 Change this line to switch datasets

# # 🧠 Auto-configured values:
model_name = dataset_model_map[app_id]

# load ai loader
ai_dataset_loader = AIDatasetLoader(
    feature_values_df=values_df,
    metadata_df=metadata_df,
    AI_predictions_df=prediction_df
)

# Load decision tree
dt_df = pd.read_csv(os.path.join(data_dir, 'decision_tree.csv'))
# dt_exp = DecisionTreeInterpreter(dt_df, metadata_df, app_id, model_name, depth=2)
# dt_exp.print_tree(as_name=True)


# Load linear model
lr_df = pd.read_csv(os.path.join(data_dir, 'logistic_regression.csv'))
lr_exp = LogisticRegressionInterpreter(lr_df, metadata_df, app_id, model_name, variant="sparse")

In [ ]:
lr_exp 

In [8]:

import importlib
import src.cognitive_models.memory as memory
importlib.reload(memory)
import src.dt_memory as dt_memory
importlib.reload(dt_memory)
import src.heuristic_lr_model as heuristic_lr_model
importlib.reload(heuristic_lr_model)
import src.lr_memory as lr_memory
importlib.reload(lr_memory)

from src.cognitive_models.memory import DeclarativeMemory, CombinedMemory
from src.dt_memory import (
    add_dt_to_memory, dt_traverse, refresh_dt_path_in_memory
)
from src.heuristic_lr_model import (
    add_lr_heuristic_to_memory, lr_heuristic, refresh_lr_heuristic_in_memory
)
from src.lr_memory import (
    add_lr_calculation_to_memory, lr_calculation, refresh_lr_calculation_in_memory
)

from typing import Optional
import random
from dataclasses import replace

In [ ]:
# ====== Gymnasium Env: LR-Calc Feature-Selection
# —— dataset- & complexity-specific retrievers for (loader, explainer) pairs ——
# Notes:
# • Action = MultiBinary(max_features) (feature mask)
# • Observation includes the *past* feature mask (length=max_features)
# • At reset(), the env picks a dataset_id (random or via options["dataset_id"])
#   and a complexity ∈ {"low","high"}; it then retrieves the matching (loader, explainer).
# • Only the LR calculation strategy is used; memory refresh happens only when with_xai=True.

import numpy as np
from collections import deque
from typing import Dict, List, Optional, Any

import gymnasium as gym
from gymnasium import spaces

# --- You must provide these from your codebase ---
# from your_module import (
#     DeclarativeMemory, CombinedMemory,
#     lr_calculation, add_lr_calculation_to_memory, refresh_lr_calculation_in_memory
# )

class LRCalcFeatureSelectEnv(gym.Env):
    """
    Constructor expects `datasets` shaped like:
        datasets = {
            "<dataset_id_A>": {
                "low":  {"loader": <LoaderA_low>,  "explainer": <ExplainerA_low>},
                "high": {"loader": <LoaderA_high>, "explainer": <ExplainerA_high>},
            },
            "<dataset_id_B>": {
                "low":  {"loader": <LoaderB_low>,  "explainer": <ExplainerB_low>},
                "high": {"loader": <LoaderB_high>, "explainer": <ExplainerB_high>},
            },
            ...
        }

    Observation vector (order):
      [ chi_norm,
        8 * normalized cognitive params
          (T_enc, T_op, retrieval_threshold, latency_factor, ddm_a, ddm_s, lapse, compute_sf),
        with_xai_flag,
        accumulated_accuracy,
        5 * past_correctness_history,
        last_reward,
        last_prob_correct,
        last_pred_time,
        max_features * past_feature_mask
      ]

    Reward:
        r = p_true - (chi * time_penalty_scale) * pred_time
    """

    metadata = {"render_modes": ["human"]}

    # Bounds per spec
    BOUNDS = {
        "T_enc": (0.05, 10.0),
        "T_op": (0.05, 1.0),
        "retrieval_threshold": (-2.0, -0.1),
        "latency_factor": (0.3, 3.0),
        "ddm_a": (0.5, 3.0),
        "ddm_s": (0.5, 2.0),
        "lapse": (0.0, 0.3),
        "compute_sf": (1.0, 3.0),
    }
    COG_PARAM_ORDER = [
        "T_enc", "T_op", "retrieval_threshold", "latency_factor",
        "ddm_a", "ddm_s", "lapse", "compute_sf"
    ]

    def __init__(
        self,
        *,
        datasets: Dict[str, Dict[str, Dict[str, Any]]],  # {dataset_id: {"low":{"loader","explainer"},"high":{...}}}
        instances_per_episode: int = 64,
        max_features: int = 6,
        chi_low: float = 0.0,
        chi_high: float = 0.03,
        xai_trial_ratio: float = 0.5,
        complexity_high_ratio: float = 0.5,   # P(episode complexity = "high")
        fixed_cog_params: Optional[Dict[str, float]] = None,
        time_penalty_scale: float = 1.0,
        instance_id_pool: Optional[List[int]] = None,
        seed: Optional[int] = None,
    ):
        super().__init__()
        if not datasets:
            raise ValueError("`datasets` mapping is required and cannot be empty.")
        self.datasets = datasets
        self.dataset_ids = list(datasets.keys())

        self.rng = np.random.default_rng(seed)
        self.instances_per_episode = int(instances_per_episode)
        self.max_features = int(max_features)
        self.chi_low, self.chi_high = float(chi_low), float(chi_high)
        self.xai_trial_ratio = float(xai_trial_ratio)
        self.complexity_high_ratio = float(np.clip(complexity_high_ratio, 0.0, 1.0))
        self.fixed_cog_params = fixed_cog_params or {}
        self.time_penalty_scale = float(time_penalty_scale)
        self.instance_id_pool = instance_id_pool if instance_id_pool is not None else list(range(1, 400))

        # Selected per episode
        self.dataset_id: Optional[str] = None
        self.episode_complexity: str = "low"    # "low" | "high"
        self.ai_dataset_loader = None
        self.explainer = None

        # Action: choose active features
        self.action_space = spaces.MultiBinary(self.max_features)

        # Observation space (see class docstring)
        len_base = 1                 # chi_norm
        len_cogs = 8                 # normalized cogs
        len_flags = 1                # with_xai flag
        len_acc = 1                  # accumulated_accuracy
        len_hist = 5                 # correctness history
        len_tail = 3                 # last_reward, last_prob_correct, last_pred_time
        len_mask = self.max_features # past feature-selection mask
        obs_len = len_base + len_cogs + len_flags + len_acc + len_hist + len_tail + len_mask

        low = np.zeros(obs_len, dtype=np.float32)
        high = np.ones(obs_len, dtype=np.float32)
        # last_reward (start of tail)
        last_reward_idx = len_base + len_cogs + len_flags + len_acc + len_hist
        low[last_reward_idx] = -1.0
        high[last_reward_idx] = 1.0
        # last_pred_time (tail index +2)
        last_pred_time_idx = last_reward_idx + 2
        low[last_pred_time_idx] = 0.0
        high[last_pred_time_idx] = 30.0

        self.observation_space = spaces.Box(low=low, high=high, dtype=np.float32)

        # Episode state
        self.step_idx = 0
        self.curr_chi = 0.0
        self.with_xai_schedule = np.zeros(self.instances_per_episode, dtype=np.int32)
        self.memory = None

        # Data buffers
        self.X_raw = None
        self.X_norm = None
        self.y = None

        # Stats
        self.accumulated_accuracy = 0.0
        self.correct_history = deque(maxlen=len_hist)
        self.last_reward = 0.0
        self.last_prob_correct = 0.0
        self.last_pred_time = 0.0
        self.last_feature_mask = np.zeros(self.max_features, dtype=np.float32)

        # Current cognitive params
        self.cog_params: Dict[str, float] = {}

    # ---------------- Helpers ----------------
    @staticmethod
    def _normalize(v: float, lo: float, hi: float) -> float:
        return float((v - lo) / (hi - lo + 1e-12))

    def _normalize_cogs(self, params: Dict[str, float]) -> List[float]:
        return [self._normalize(params[k], *self.BOUNDS[k]) for k in self.COG_PARAM_ORDER]

    def _choose_complexity(self) -> str:
        return "high" if self.rng.random() < self.complexity_high_ratio else "low"

    def _pick_dataset_and_bind(self, *, dataset_id: Optional[str], complexity: str):
        """
        Choose a dataset_id (random if None) and bind loader/explainer
        for the requested complexity ("low" or "high").
        """
        if dataset_id is None:
            dataset_id = self.rng.choice(self.dataset_ids).item() if hasattr(self.rng.choice(self.dataset_ids), "item") else self.rng.choice(self.dataset_ids)
        if dataset_id not in self.datasets:
            raise KeyError(f"dataset_id '{dataset_id}' not found in provided datasets.")
        if complexity not in self.datasets[dataset_id]:
            raise KeyError(f"complexity '{complexity}' not available for dataset_id '{dataset_id}'.")

        entry = self.datasets[dataset_id][complexity]
        if not isinstance(entry, dict) or "loader" not in entry or "explainer" not in entry:
            raise ValueError(f"datasets['{dataset_id}']['{complexity}'] must be a dict with 'loader' and 'explainer'.")

        self.dataset_id = dataset_id
        self.ai_dataset_loader = entry["loader"]
        self.explainer = entry["explainer"]

    def _initialize_memory(self):
        dm = DeclarativeMemory(
            retrieval_threshold=self.cog_params["retrieval_threshold"],
            latency_factor=self.cog_params["latency_factor"],
            latency_exponent=0.5,
            max_assoc_strength=2.0,
            mismatch_penalty=-1.0,
            activation_noise=0.3,
            decay=0.5,
        )
        self.memory = CombinedMemory(dm, wm_capacity=7)
        add_lr_calculation_to_memory(self.explainer, self.memory)
        self.memory.tick(10)

    def _build_obs(self, with_xai_flag: int) -> np.ndarray:
        chi_norm = self.curr_chi / max(self.chi_high, 1e-9)
        cog_norm = self._normalize_cogs(self.cog_params)
        hist = list(self.correct_history)
        if len(hist) < self.correct_history.maxlen:
            hist += [0.0] * (self.correct_history.maxlen - len(hist))

        obs = np.array(
            [chi_norm] +
            cog_norm +
            [float(with_xai_flag)] +
            [float(self.accumulated_accuracy)] +
            [float(h) for h in hist] +
            [float(self.last_reward), float(self.last_prob_correct), float(self.last_pred_time)],
            dtype=np.float32
        )
        obs = np.concatenate([obs, self.last_feature_mask.astype(np.float32)], dtype=np.float32)
        return obs

    def _sample_cog_params(self) -> Dict[str, float]:
        params = {}
        for k in self.COG_PARAM_ORDER:
            if k in self.fixed_cog_params and isinstance(self.fixed_cog_params[k], (int, float)):
                params[k] = float(self.fixed_cog_params[k])
            else:
                lo, hi = self.BOUNDS[k]
                params[k] = float(self.rng.uniform(lo, hi))
        # Snap compute_sf to {1,2,3}
        lo_cs, hi_cs = self.BOUNDS["compute_sf"]
        params["compute_sf"] = int(np.clip(round(params["compute_sf"]), lo_cs, hi_cs))
        return params

    # ---------------- Gymnasium API ----------------
    def reset(self, *, seed: Optional[int] = None, options: Optional[dict] = None):
        if seed is not None:
            self.rng = np.random.default_rng(seed)

        # 1) Choose complexity and dataset_id (respect options if provided)
        self.episode_complexity = self._choose_complexity()
        ds_id_opt = None
        if options and isinstance(options, dict) and "dataset_id" in options:
            ds_id_opt = str(options["dataset_id"])
        self._pick_dataset_and_bind(dataset_id=ds_id_opt, complexity=self.episode_complexity)

        # 2) Build with-XAI schedule
        n = self.instances_per_episode
        n_xai = int(round(n * self.xai_trial_ratio))
        flags = np.array([1] * n_xai + [0] * (n - n_xai), dtype=np.int32)
        self.rng.shuffle(flags)
        self.with_xai_schedule = flags

        # 3) Cognitive params & memory
        self.cog_params = self._sample_cog_params()
        self._initialize_memory()

        # 4) Episode chi
        self.curr_chi = float(self.rng.uniform(self.chi_low, self.chi_high))

        # 5) Data for this episode from the chosen loader
        idx = self.rng.choice(self.instance_id_pool, size=self.instances_per_episode, replace=False).tolist()
        inst_raw, labels = self.ai_dataset_loader.load_instances(idx, normalize=False)
        inst_norm, _ = self.ai_dataset_loader.load_instances(idx, normalize=True)
        self.X_raw = np.asarray(inst_raw, dtype=np.float32)
        self.X_norm = np.asarray(inst_norm, dtype=np.float32)  # kept if you later need normalized
        self.y = np.asarray(labels, dtype=np.int64)

        # 6) Reset stats
        self.step_idx = 0
        self.accumulated_accuracy = 0.0
        self.correct_history.clear()
        self.last_reward = 0.0
        self.last_prob_correct = 0.0
        self.last_pred_time = 0.0
        self.last_feature_mask = np.zeros(self.max_features, dtype=np.float32)

        obs = self._build_obs(with_xai_flag=int(self.with_xai_schedule[0]))
        info = {
            "dataset_id": self.dataset_id,
            "episode_complexity": self.episode_complexity,
            "cog_params": self.cog_params.copy(),
            "chi": self.curr_chi,
        }
        return obs, info

    def step(self, action):
        action = np.asarray(action, dtype=np.int64).flatten()
        if action.size != self.max_features:
            raise ValueError(f"Expected action of size {self.max_features}, got {action.size}")
        active_indices = [i for i, b in enumerate(action.tolist()) if b == 1]

        terminated = False
        truncated = self.step_idx >= self.instances_per_episode
        if truncated:
            return self._build_obs(with_xai_flag=0), 0.0, terminated, truncated, {}

        with_xai = bool(self.with_xai_schedule[self.step_idx])

        x_raw = self.X_raw[self.step_idx]
        y_true = int(self.y[self.step_idx])

        T_enc = float(self.cog_params["T_enc"])
        T_op = float(self.cog_params["T_op"])
        ddm_a = float(self.cog_params["ddm_a"])
        ddm_s = float(self.cog_params["ddm_s"])
        compute_sf = int(self.cog_params["compute_sf"])
        lapse = float(self.cog_params["lapse"])

        # Run LR calculation (mask inside lr_calculation if supported)
        probs, pred_time, _ = lr_calculation(
            x_raw, self.memory, lr_exp=self.explainer,
            T_enc=T_enc, T_op=T_op, ddm_a=ddm_a, ddm_s=ddm_s,
            compute_sf=compute_sf,
            mode=("read" if with_xai else "retrieve"),
            # active_indices=active_indices,  # uncomment if your lr_calculation supports feature masking
        )

        p_true = float(probs[y_true])
        if lapse > 0.0:
            p_true = (1.0 - lapse) * p_true + 0.5 * lapse

        reward = p_true - (self.curr_chi * self.time_penalty_scale) * float(pred_time)

        if with_xai:
            refresh_lr_calculation_in_memory(
                self.memory, self.explainer,
                intercept_display_sf=int(compute_sf),
                factor_display_sf=int(compute_sf),
            )

        correct = 1.0 if p_true > 0.5 else 0.0
        self.correct_history.append(correct)
        self.accumulated_accuracy = (
            (self.accumulated_accuracy * self.step_idx + correct) / (self.step_idx + 1)
        )

        self.step_idx += 1
        truncated = self.step_idx >= self.instances_per_episode

        # Update past feature mask for NEXT observation
        self.last_feature_mask = action.astype(np.float32)

        self.last_reward = float(reward)
        self.last_prob_correct = float(p_true)
        self.last_pred_time = float(pred_time)

        next_with_xai_flag = int(self.with_xai_schedule[self.step_idx - 1] if truncated
                                 else self.with_xai_schedule[self.step_idx])
        obs = self._build_obs(with_xai_flag=next_with_xai_flag)

        info = {
            "dataset_id": self.dataset_id,
            "episode_complexity": self.episode_complexity,
            "active_indices": active_indices,
            "with_xai": with_xai,
            "prob_correct": float(p_true),
            "pred_time": float(pred_time),
            "y_true": y_true,
            "cog_params": self.cog_params.copy(),
            "chi": self.curr_chi,
        }
        return obs, float(reward), terminated, truncated, info

    def render(self): pass
    def close(self): pass
